# Part 3 - Price-History Analytics (4-credit only)

> Case study part: Part 3 of 3
> Required for: 4-credit (BDI 593) students only
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

In Parts 1 and 2 we treated each listing as a single, static row. But Zillow
also publishes a price-history event log - every list, price change,
contingent, pending, sold, and removal event for each property. That event
log is what an experienced real-estate analyst uses to answer the questions
that actually drive acquisitions:

- How long do homes typically take to sell in each market?
- How often do sellers cut their list price, and do listings with price cuts
  sell less often?
- What is the typical gap between the list price and the sale price?
- Given the list price, how well can we predict the realized sale price, and
  can a model beat the simple rule "sale price = list price"?

Part 2 predicted `price`, the most recent sold price, from structural and
text features alone. This notebook adds the one thing Part 2 did not have:
the list price of the listing that ended in that sale. We reshape the event
log into one row per listing cycle, describe market dynamics, and then
measure what knowing the list price adds, both for the sale price and for the
list-to-sale discount.

## ⚙️ Setup


In [1]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "simple_white"

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 80)

## 📥 Load All Three Inputs

- `properties_clean.parquet` from Part 1
- `nlp_features.parquet` from Part 2b
- `price_history.parquet` (raw, this folder)


In [2]:
DATA_DIR = "data"

properties = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
nlp_pack = pd.read_parquet(f"{DATA_DIR}/nlp_features.parquet")
history = pd.read_parquet(f"{DATA_DIR}/price_history.parquet")

print("properties:", properties.shape)
print("nlp_pack:  ", nlp_pack.shape)
print("history:   ", history.shape)
history.head()


properties: (35273, 33)
nlp_pack:   (35273, 19)
history:    (291925, 5)


,property_id,date,event_type,price,price_change_rate
0,3197116,2021-05-21,listedForSale,156000.0,NaN
1,3197116,2021-06-05,pendingSale,156000.0,NaN
2,3197116,2021-07-26,sold,160000.0,0.025641
3,3197116,2023-10-16,listedForSale,189900.0,0.186875
4,3197116,2023-10-29,contingent,189900.0,NaN


## 🩺 What Is in the Event Log?


In [3]:
event_counts = history["event_type"].value_counts().sort_values().reset_index()
event_counts.columns = ["event_type", "count"]
fig = px.bar(
    event_counts,
    x="count",
    y="event_type",
    orientation="h",
    title="Volume of each event type in the Zillow price history",
    labels={"count": "Events recorded", "event_type": ""},
    height=400,
)
fig.show()


:::{seealso} Event-type cheat sheet

- `listedForSale` - a fresh listing.
- `priceChange` - seller cut or raised the asking price.
- `contingent` - offer accepted, contingent on financing/inspection/etc.
- `pendingSale` - under contract, sale expected.
- `sold` - closed sale.
- `listingRemoved` - pulled from the market without selling.
- `listedForRent` - switched from for-sale to for-rent (signal for
  unsuccessful sale).

:::

## 🔁 Reconstruct Listing Cycles per Property

Many properties have been listed and sold more than once. We define a
"cycle" as the events from one `listedForSale` row up to the next
`listedForSale` row for the same property. Most cycles end in a `sold` or a
`listingRemoved` event; some have no recorded outcome.

Number each cycle by counting fresh listings per property.


In [4]:
hist = history.sort_values(["property_id", "date"]).copy()

hist["is_listing"] = (hist["event_type"] == "listedForSale").astype(int)
hist["cycle_id"] = hist.groupby("property_id")["is_listing"].cumsum()

# Events before a property's first listing on record belong to no cycle.
before_first = hist["cycle_id"] == 0
print(
    f"Events before the first listing on record: {before_first.sum():,} "
    f"({before_first.mean():.1%} of all events)"
)
print(hist.loc[before_first, "event_type"].value_counts(normalize=True).round(2).to_string())
hist = hist[~before_first]

Events before the first listing on record: 38,880 (13.3% of all events)
event_type
sold              0.72
listingRemoved    0.11
listedForRent     0.10
priceChange       0.04
pendingSale       0.02
contingent        0.00


About one event in eight happens before the property's first listing on
record. Most of these are older `sold` records with no listing in the log, so
they cannot be paired with a list price. We drop them. That is a real loss of
history, not a rounding error, so keep it in mind when you quote counts.

Distribution of cycles per property.


In [5]:
cycles_n = hist.groupby("property_id")["cycle_id"].max()
cycle_counts = cycles_n.clip(upper=5).value_counts().sort_index().reset_index()
cycle_counts.columns = ["cycles", "properties"]
cycle_counts["cycles"] = cycle_counts["cycles"].astype(str).replace({"5": "5+"})

fig = px.bar(
    cycle_counts,
    x="cycles",
    y="properties",
    title="Number of listing cycles per property (5 or more grouped)",
    labels={"cycles": "Cycles per property", "properties": "Properties"},
)
fig.show()

## 🧮 Aggregate Each Cycle Into One Row

For every cycle, we want:

- the original list price and date,
- whether and when it sold, and how many `sold` events it contains,
- the realized sale price,
- the list-to-sale discount,
- how many price cuts happened during the cycle,
- and the cumulative depth of those price cuts.

Looping a Python function over 70,000 cycles is slow. Instead we summarize
each event type with one grouped aggregation and join the pieces. Every cycle
starts with exactly one `listedForSale` row, so that table defines the cycles.

In [6]:
keys = ["property_id", "cycle_id"]
event = hist["event_type"]

listed = (
    hist[event == "listedForSale"]
    .groupby(keys)
    .agg(list_date=("date", "first"), list_price=("price", "first"))
)
sold = (
    hist[event == "sold"]
    .groupby(keys)
    .agg(
        sold_date=("date", "last"),
        sold_price=("price", "last"),
        n_sold_events=("date", "size"),
    )
)
pending = hist[event == "pendingSale"].groupby(keys).agg(pending_date=("date", "first"))
cuts = (
    hist[(event == "priceChange") & (hist["price_change_rate"] < 0)]
    .groupby(keys)
    .agg(n_price_cuts=("date", "size"), total_pct_cut=("price_change_rate", "sum"))
)
removed = hist[event == "listingRemoved"].groupby(keys).size().rename("n_removed")

Join the pieces into one row per cycle and compute the realized list-to-sale
change.

In [7]:
cycle_df = listed.join([sold, pending, cuts, removed]).reset_index()

cycle_df["n_sold_events"] = cycle_df["n_sold_events"].fillna(0).astype(int)
cycle_df["n_price_cuts"] = cycle_df["n_price_cuts"].fillna(0).astype(int)
cycle_df["total_pct_cut"] = cycle_df["total_pct_cut"].fillna(0) * 100  # negative
cycle_df["days_to_pending"] = (cycle_df["pending_date"] - cycle_df["list_date"]).dt.days
cycle_df["days_to_sold"] = (cycle_df["sold_date"] - cycle_df["list_date"]).dt.days
cycle_df["ended_sold"] = (cycle_df["n_sold_events"] > 0).astype(int)
cycle_df["ended_removed"] = (
    cycle_df["n_removed"].notna() & (cycle_df["ended_sold"] == 0)
).astype(int)
cycle_df["list_to_sale_pct"] = (
    (cycle_df["sold_price"] - cycle_df["list_price"]) / cycle_df["list_price"] * 100
)
cycle_df = cycle_df.drop(columns=["pending_date", "n_removed"])

print(f"{len(cycle_df):,} listing cycles reconstructed")
cycle_df.head()

70,549 listing cycles reconstructed


,property_id,cycle_id,list_date,list_price,sold_date,sold_price,n_sold_events,n_price_cuts,total_pct_cut,days_to_pending,days_to_sold,ended_sold,ended_removed,list_to_sale_pct
0,3197116,1,2021-05-21,156000.0,2021-07-26,160000.0,1,0,0.000000,15.0,66.0,1,0,2.564103
1,3197116,2,2023-10-16,189900.0,2023-12-22,181000.0,1,0,0.000000,45.0,67.0,1,0,-4.686677
2,3197128,1,2023-10-19,145000.0,2023-12-18,139000.0,1,1,-3.448966,29.0,60.0,1,0,-4.137931
3,3197135,1,2019-12-06,94500.0,NaT,NaN,0,1,-5.291005,87.0,NaN,0,0,NaN
4,3197135,2,2020-03-06,89500.0,2020-04-15,60000.0,1,1,-17.318436,11.0,40.0,1,0,-32.960894


## 📈 Market Dynamics - Visualizations for the Deck


Not every sold cycle is a clean list-then-sell pair. Some cycles contain two
`sold` events (the home changed hands again with no new listing on record),
and a few pair a list price with a sale price that cannot belong to the same
transaction. We keep a sold cycle only if:

1. it has a list price and a sale price above \$10,000 and sold within two
   years of listing,
2. it contains exactly one `sold` event, and
3. the sale price is between 0.5 and 1.5 times the list price.

Rule 3 is deliberately loose. It removes pairs that cannot be the same deal,
not unusual but real ones. Then we join in market and campus context.

In [8]:
sold_cycles = cycle_df.dropna(subset=["sold_price", "list_price", "days_to_sold"])
sold_cycles = sold_cycles[
    (sold_cycles["days_to_sold"].between(0, 365 * 2))
    & (sold_cycles["list_price"] > 10_000)
    & (sold_cycles["sold_price"] > 10_000)
].copy()
print(f"{len(sold_cycles):,} sold cycles with a list price and a sale within 2 years")
print(
    "List-to-sale change before the plausibility rules: "
    f"{sold_cycles['list_to_sale_pct'].min():.1f}% to "
    f"{sold_cycles['list_to_sale_pct'].max():,.1f}%"
)

one_sale = sold_cycles["n_sold_events"] == 1
ratio = sold_cycles["sold_price"] / sold_cycles["list_price"]
plausible = ratio.between(0.5, 1.5)
print(f"  dropped, more than one sold event:      {(~one_sale).sum():,}")
print(f"  dropped, sale outside 0.5x-1.5x list:   {(one_sale & ~plausible).sum():,}")
sold_cycles = sold_cycles[one_sale & plausible]

market_lookup = properties[
    ["property_id", "market", "campus", "address_state"]
].drop_duplicates("property_id")
sold_cycles = sold_cycles.merge(market_lookup, on="property_id", how="inner")
print(f"{len(sold_cycles):,} plausible sold cycles with market context")

50,723 sold cycles with a list price and a sale within 2 years
List-to-sale change before the plausibility rules: -99.9% to 2,854.0%
  dropped, more than one sold event:      629
  dropped, sale outside 0.5x-1.5x list:   254
48,603 plausible sold cycles with market context


:::{note} Box plots from summary statistics

A Plotly box plot stores every data point inside the notebook, and we have
tens of thousands of cycles. As in Part 1, we compute the five numbers each
box needs per market with pandas and draw the boxes from those.

:::

In [9]:
def box_stats(data, by, value):
    """Quartiles and 1.5 x IQR whiskers of `value` for each group in `by`."""

    def five_numbers(s):
        q1, median, q3 = s.quantile([0.25, 0.5, 0.75])
        iqr = q3 - q1
        return pd.Series(
            {
                "q1": q1,
                "median": median,
                "q3": q3,
                "lower": s[s >= q1 - 1.5 * iqr].min(),
                "upper": s[s <= q3 + 1.5 * iqr].max(),
            }
        )

    return data.groupby(by)[value].apply(five_numbers).unstack().reset_index()


def box_from_stats(stats, by, title, x_title, y_title, **layout):
    fig = go.Figure(
        go.Box(
            x=stats[by],
            q1=stats["q1"],
            median=stats["median"],
            q3=stats["q3"],
            lowerfence=stats["lower"],
            upperfence=stats["upper"],
            name="",
        )
    )
    fig.update_layout(title=title, xaxis_title=x_title, yaxis_title=y_title, **layout)
    return fig

In [10]:
days_box = box_stats(sold_cycles, "market", "days_to_sold").sort_values("market")

fig = box_from_stats(
    days_box,
    "market",
    title="Days from list to sale, by market",
    x_title="",
    y_title="Days from list to sale",
    height=500,
)
fig.update_yaxes(range=[0, 300])
fig.update_xaxes(tickangle=-30)
fig.show()

In [11]:
disc_box = box_stats(sold_cycles, "market", "list_to_sale_pct").sort_values("market")

fig = box_from_stats(
    disc_box,
    "market",
    title="List-to-sale price change, by market (%)",
    x_title="",
    y_title="(sale - list) / list × 100%",
    height=500,
)
fig.add_hline(y=0, line_dash="dot")
fig.update_yaxes(range=[-30, 30])
fig.update_xaxes(tickangle=-30)
fig.show()

The same dynamics as a table, one row per market.

In [12]:
print(
    f"Footprint: median {sold_cycles['days_to_sold'].median():.0f} days from list "
    f"to sale, median list-to-sale change {sold_cycles['list_to_sale_pct'].median():.1f}%"
)

market_summary = (
    sold_cycles.groupby("market")
    .agg(
        sold_cycles=("property_id", "size"),
        median_days_to_sale=("days_to_sold", "median"),
        median_list_to_sale_pct=("list_to_sale_pct", "median"),
        pct_sold_above_list=("list_to_sale_pct", lambda s: (s > 0).mean() * 100),
        pct_with_price_cut=("n_price_cuts", lambda s: (s > 0).mean() * 100),
    )
    .sort_values("median_days_to_sale")
    .round(1)
)
market_summary

Footprint: median 55 days from list to sale, median list-to-sale change -1.6%


,sold_cycles,median_days_to_sale,median_list_to_sale_pct,pct_sold_above_list,pct_with_price_cut
market,,,,,
"Davis, CA",1150,40.0,-0.5,38.5,26.4
"Lafayette, IN",5111,45.0,-1.4,27.1,27.3
"Carrboro, NC",217,46.0,-0.2,41.0,28.1
"Ann Arbor, MI",4068,46.0,-0.5,38.9,25.0
"Madison, WI",7145,50.0,0.0,46.2,23.9
"West Lafayette, IN",2317,50.0,-1.6,24.3,28.0
"Chapel Hill, NC",3448,55.0,-1.3,30.7,28.3
"Athens, GA",3447,56.0,-2.5,19.6,32.6
"State College, PA",1440,57.0,-1.1,29.5,26.1


:::{tip} Investor reading

Read the list-to-sale box against the zero line. A median at or above zero
means homes typically sell at or above the asking price (a seller's market); a
median below zero means most homes sell at a discount (a buyer's market). In
this footprint no market's median is above zero. Madison sits at zero, while
Gainesville (about -3%) and Athens (about -2.5%) show the deepest typical
discounts and the highest share of cycles with a price cut.

:::

Do price cuts signal a listing that will not sell? To answer that we need the
cycles that did not sell too, so we go back to `cycle_df` (all cycles, not
only the sold ones) and keep every cycle with a recorded outcome: a sale or a
removal.

In [13]:
outcome = cycle_df[(cycle_df["ended_sold"] == 1) | (cycle_df["ended_removed"] == 1)].copy()
outcome["cuts"] = outcome["n_price_cuts"].clip(upper=4).astype(str).replace({"4": "4+"})
print(f"{len(outcome):,} of {len(cycle_df):,} cycles ended in a sale or a removal")

cut_effect = outcome.groupby("cuts", as_index=False).agg(
    cycles=("ended_sold", "size"), pct_sold=("ended_sold", "mean")
)
cut_effect["pct_sold"] = (cut_effect["pct_sold"] * 100).round(1)

fig = px.bar(
    cut_effect,
    x="cuts",
    y="pct_sold",
    text="pct_sold",
    title="Share of listing cycles that ended in a sale, by number of price cuts",
    labels={"cuts": "Price cuts during the cycle", "pct_sold": "% of cycles that sold"},
)
fig.update_yaxes(range=[0, 100])
fig.show()

cut_effect

64,926 of 70,549 cycles ended in a sale or a removal


,cuts,cycles,pct_sold
0,0,44496,86.6
1,1,10922,77.8
2,2,5049,76.4
3,3,2303,77.6
4,4+,2156,77.7


Cycles with no price cut ended in a sale about 87% of the time. Cycles with at
least one cut sold about 76% to 78% of the time, and further cuts do not lower
that rate. So a first price cut is a warning sign, but the number of cuts
beyond the first tells us little.

Two caveats. This is an association, not the effect of cutting: sellers cut
because a listing is not attracting offers. And the number of cuts is only
known while the listing is live, so it cannot be a feature in a model that
predicts the outcome at list time.

Quarterly sale volume to confirm we have enough recent data to model.


In [14]:
sold_cycles["sold_quarter"] = (
    sold_cycles["sold_date"].dt.to_period("Q").dt.to_timestamp()
)
ts = sold_cycles.groupby("sold_quarter", as_index=False).size()
fig = px.line(
    ts,
    x="sold_quarter",
    y="size",
    title="Closed sales per quarter (campustown footprint)",
    labels={"sold_quarter": "", "size": "Closed sales"},
    markers=True,
)
fig.show()


Closed sales jump in the second quarter of 2023 and stay at roughly 1,600 to
3,400 per quarter after that, with a spring peak each year. Earlier years are
much thinner: every property in the extract has a recorded sale from 2023 on,
so older sales appear only as earlier cycles of the same homes. The modeling
window below (sales from 2024 on) is well covered. The last quarter ends at
the extract date in late March 2026.

## 🎯 Re-frame the Modeling Problem

In Part 2 we predicted `price`, the most recent sold price, from a home's
structure, location, and description. The modeling frame below uses the same
sales: for almost every row, `sold_price` equals Part 2's `price` (we check
this below). So the target has not changed. What changes is the information
the model may use: we now know the list price of the listing that led to the
sale.

That turns the question into: given the list price, how well can we predict
the realized sale price? The natural benchmark is the naive rule "sale price
= list price". A model is only useful to QCP if it beats that rule.

We will:

1. Take only the most recent plausible sold cycle per property (so each
   property appears at most once).
2. Join in the structured features from Part 1 and the NLP features from
   Part 2b.
3. Engineer history-derived features that would have been observable
   at the moment of listing (so we don't leak future information).
4. Compare naive rules and models on the same held-out test set, with and
   without the list price.

Keep only the most recent sold cycle per property.


In [15]:
latest_sold = (
    sold_cycles.sort_values(["property_id", "sold_date"])
    .groupby("property_id", as_index=False)
    .tail(1)
)
print(f"{len(latest_sold):,} properties with a most-recent sold cycle")


30,147 properties with a most-recent sold cycle


:::{caution} Avoiding target leakage

`days_to_sold`, `n_price_cuts`, and `total_pct_cut` are only known after the
listing has been on the market for a while, and the outcome is only known
after it closes. If we feed them into a model that predicts the sale price of
a new listing, we are cheating - those features are unobservable at list
time.

Instead, we engineer historical priors per market from sales that closed
before the modeling period:

- the historical median list-to-sale change in this property's market,
- the historical median days-to-sale in this property's market,
- the historical average number of price cuts in this property's market.

These are all observable at list time and are honest features. So is the
list price itself.

:::

Split the cycles into a historical "prior" pool and a recent "target" pool,
then compute per-market priors from the historical pool only.


In [16]:
HORIZON = pd.Timestamp("2024-01-01")

prior_pool = sold_cycles[sold_cycles["sold_date"] < HORIZON]
target_pool = latest_sold[latest_sold["sold_date"] >= HORIZON].copy()

market_priors = (
    prior_pool.groupby("market")
    .agg(
        market_hist_disc=("list_to_sale_pct", "median"),
        market_hist_days=("days_to_sold", "median"),
        market_hist_cuts=("n_price_cuts", "mean"),
    )
    .reset_index()
)
target_pool = target_pool.merge(market_priors, on="market", how="left")
print(f"Prior pool: {len(prior_pool):,} historical cycles")
print(f"Target pool (>= {HORIZON.date()}): {len(target_pool):,} cycles to model")
target_pool.head(3)


Prior pool: 26,461 historical cycles
Target pool (>= 2024-01-01): 21,648 cycles to model


,property_id,cycle_id,list_date,list_price,sold_date,sold_price,n_sold_events,n_price_cuts,total_pct_cut,days_to_pending,days_to_sold,ended_sold,ended_removed,list_to_sale_pct,market,campus,address_state,sold_quarter,market_hist_disc,market_hist_days,market_hist_cuts
0,3197135,3,2025-07-15,190250.0,2025-08-15,175000.0,1,1,-8.015769,21.0,31.0,1,0,-8.015769,"Champaign, IL",University of Illinois Urbana-Champaign,IL,2025-07-01,-3.004587,68.0,0.670633
1,3197137,2,2024-02-07,175000.0,2024-03-29,165000.0,1,0,0.000000,NaN,51.0,1,0,-5.714286,"Champaign, IL",University of Illinois Urbana-Champaign,IL,2024-01-01,-3.004587,68.0,0.670633
2,3197168,1,2024-07-19,170000.0,2024-08-29,165000.0,1,0,0.000000,5.0,41.0,1,0,-2.941176,"Champaign, IL",University of Illinois Urbana-Champaign,IL,2024-07-01,-3.004587,68.0,0.670633


## 🧱 Assemble the Final Modeling Frame


Pull the structural features from Part 1. We deliberately drop
`market`, `campus`, and `address_state` here because `target_pool` already
carries them via the earlier `market_lookup` merge - if we let the same
columns appear on both sides of this merge they would get suffixed to
`address_state_x` / `address_state_y` and break later feature lookups.


In [17]:
struct_cols = [
    "property_id",
    "beds",
    "baths",
    "living_area",
    "lot_size",
    "year_built",
    "reso_parking_capacity",
    "latitude",
    "longitude",
    "address_city",
    "address_zipcode",
    "reso_architectural_style",
]
struct = properties[struct_cols].drop_duplicates("property_id").copy()
struct["address_zipcode"] = struct["address_zipcode"].astype(str)


In [18]:
frame = target_pool.merge(struct, on="property_id", how="inner").merge(
    nlp_pack, on="property_id", how="left"
)
print(f"{len(frame):,} rows in the final modeling frame")
print("columns:", list(frame.columns)[:14], "...")

# Is this the same sale that Part 2 used as its target?
part2_price = frame["property_id"].map(properties.set_index("property_id")["price"])
print(f"sold_price equals Part 2's price for {(frame['sold_price'] == part2_price).mean():.1%} of rows")

21,648 rows in the final modeling frame
columns: ['property_id', 'cycle_id', 'list_date', 'list_price', 'sold_date', 'sold_price', 'n_sold_events', 'n_price_cuts', 'total_pct_cut', 'days_to_pending', 'days_to_sold', 'ended_sold', 'ended_removed', 'list_to_sale_pct'] ...
sold_price equals Part 2's price for 98.9% of rows


Define the numeric and categorical feature lists for the model.


In [19]:
NUMERIC_FEATURES = [
    # Structural
    "beds",
    "baths",
    "living_area",
    "lot_size",
    "year_built",
    "reso_parking_capacity",
    "latitude",
    "longitude",
    # Listing-time signal
    "list_price",
    # Market priors (history-derived but leakage-safe)
    "market_hist_disc",
    "market_hist_days",
    "market_hist_cuts",
    # NLP (numeric)
    "n_chars",
    "n_words",
] + [c for c in frame.columns if c.startswith("amen_")]

CATEGORICAL_FEATURES = [
    "address_state",
    "address_city",
    "address_zipcode",
    "campus",
    "reso_architectural_style",
]
TARGET = "sold_price"


Build the X / y matrices and the train/test split.


In [20]:
X = frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = frame[TARGET].copy()
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X,
    y_log,
    y,
    test_size=0.20,
    random_state=42,
    stratify=frame["address_state"],
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


Train: 17,318   Test: 4,330


## 🔧 Naive Rules vs. Models, One Comparison Table

We compare five ways to predict the sale price, all on the same test set:

- Naive rule: sale price = list price.
- Market rule: list price × (1 + the market's historical median list-to-sale
  change).
- No-list-price model: XGBoost on structural, market-prior, and NLP features,
  without the list price. This is the Part 2 setup applied to this frame.
- List-price model: XGBoost on the list price plus structural features.
- History-aware model: the list-price model plus the market priors and the
  NLP features.

Pipeline factory shared by all the models in this notebook.

In [21]:
def make_pipeline(num_cols, cat_cols):
    num = Pipeline([("impute", SimpleImputer(strategy="median"))])
    cat = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
        ]
    )
    pre = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
    return Pipeline(
        [
            ("preprocess", pre),
            (
                "regressor",
                XGBRegressor(
                    n_estimators=600,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    random_state=42,
                    n_jobs=-1,
                    tree_method="hist",
                ),
            ),
        ]
    )


Fit the list-price model.

In [22]:
BASELINE_NUM = [
    "beds",
    "baths",
    "living_area",
    "lot_size",
    "year_built",
    "reso_parking_capacity",
    "latitude",
    "longitude",
    "list_price",
]
baseline_pipe = make_pipeline(BASELINE_NUM, CATEGORICAL_FEATURES)
baseline_pipe.fit(X_train, y_train_log)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](35,)","['beds','baths','living_area',...,'address_zipcode','campus', 'reso_architectural_style']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,35
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Th

Fit the history-aware model on the full feature set.


In [23]:
history_pipe = make_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
history_pipe.fit(X_train, y_train_log)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](35,)","['beds','baths','living_area',...,'address_zipcode','campus', 'reso_architectural_style']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,35
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Th

Fit the no-list-price model: the history-aware feature set minus the list
price.

In [24]:
NO_LIST_NUM = [c for c in NUMERIC_FEATURES if c != "list_price"]
no_list_pipe = make_pipeline(NO_LIST_NUM, CATEGORICAL_FEATURES)
no_list_pipe.fit(X_train, y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](35,)","['beds','baths','living_area',...,'address_zipcode','campus', 'reso_architectural_style']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,35
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Th

Compare the two simple rules and the three models on the held-out test set.
Besides MAPE (a mean) we report the median absolute percentage error, which
describes the typical listing.

In [25]:
def metrics(y_true, y_pred):
    ape = np.abs((y_true - y_pred) / y_true) * 100
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": ape.mean(),
        "Median APE": ape.median(),
    }


naive_pred = X_test["list_price"]
market_rule_pred = X_test["list_price"] * (1 + X_test["market_hist_disc"] / 100)
no_list_pred = np.expm1(no_list_pipe.predict(X_test))
baseline_pred = np.expm1(baseline_pipe.predict(X_test))
history_pred = np.expm1(history_pipe.predict(X_test))

comparison = pd.DataFrame(
    {
        "No list price (structural + priors + NLP)": metrics(y_test_raw, no_list_pred),
        "Naive rule: sale = list": metrics(y_test_raw, naive_pred),
        "Market rule: list x market median change": metrics(y_test_raw, market_rule_pred),
        "List price + structural": metrics(y_test_raw, baseline_pred),
        "History-aware (list + structural + priors + NLP)": metrics(
            y_test_raw, history_pred
        ),
    }
).T
comparison.round({"MAE": 0, "RMSE": 0, "R2": 3, "MAPE": 2, "Median APE": 2})

,MAE,RMSE,R2,MAPE,Median APE
No list price (structural + priors + NLP),64822.0,124084.0,0.857,12.41,8.46
Naive rule: sale = list,30157.0,67165.0,0.958,5.66,3.54
Market rule: list x market median change,28529.0,63184.0,0.963,5.36,3.16
List price + structural,27907.0,52558.0,0.974,5.34,3.55
History-aware (list + structural + priors + NLP),27826.0,52575.0,0.974,5.31,3.60


Three results stand out:

1. Knowing the list price is what matters. Without it, the model misses by
   about 12% on average (MAPE). The naive rule "sale = list" alone cuts
   that to about 5.7%.
2. The models beat the naive rule, but only modestly. MAPE falls from about
   5.7% to 5.3%, and RMSE falls from about \$67K to \$53K, so the models mainly
   fix the larger misses. The market priors and NLP features add almost
   nothing on top of the list price and the structural features.
3. For the typical listing, nothing beats the simple market rule. Its median
   error (about 3.2%) is lower than either list-price model's (about 3.6%).

For QCP, a reasonable first estimate of the sale price is the list price
adjusted by the market's typical discount. The models earn their complexity
only where that rule is far off.

## 🔬 Where Does the History-Aware Model Beat the Naive Rule?

In [26]:
diag = pd.DataFrame(
    {
        "actual": y_test_raw.values,
        "naive": naive_pred.values,
        "history": history_pred,
        "market": frame.loc[X_test.index, "market"].values,
    }
)
diag["naive_pct_err"] = (diag["actual"] - diag["naive"]).abs() / diag["actual"] * 100
diag["history_pct_err"] = (
    (diag["actual"] - diag["history"]).abs() / diag["actual"] * 100
)
diag["improvement_pp"] = diag["naive_pct_err"] - diag["history_pct_err"]

by_market = (
    diag.groupby("market", as_index=False)
    .agg(
        n=("actual", "count"),
        naive=("naive_pct_err", "mean"),
        history=("history_pct_err", "mean"),
        improvement_pp=("improvement_pp", "mean"),
    )
    .sort_values("improvement_pp", ascending=False)
)
by_market.round(2)

,market,n,naive,history,improvement_pp
7,"Gainesville, FL",687,6.61,5.26,1.35
15,"West Lafayette, IN",200,5.13,4.16,0.97
1,"Athens, GA",318,5.95,5.05,0.90
11,"Middleton, WI",83,6.36,5.74,0.61
13,"State College, PA",134,5.07,4.82,0.25
3,"Champaign, IL",265,5.24,5.05,0.19
4,"Chapel Hill, NC",336,5.54,5.39,0.15
8,"Iowa City, IA",256,4.44,4.30,0.14
5,"Charlottesville, VA",313,6.70,6.59,0.11
9,"Lafayette, IN",455,4.27,4.16,0.11


In [27]:
fig = px.bar(
    by_market.sort_values("improvement_pp"),
    x="improvement_pp",
    y="market",
    orientation="h",
    color="improvement_pp",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="History-aware model - reduction in mean |% error| vs. the naive rule",
    labels={"improvement_pp": "Δ MAPE (naive - history), pp", "market": ""},
    height=600,
)
fig.show()

The history-aware model gains most over the naive rule in Gainesville, West
Lafayette, and Athens. Gainesville and Athens are also the markets with the
deepest typical discounts in the market summary above, so "sale = list" is
most biased there and a model has the most to correct. In Urbana and Davis the
model does worse than the naive rule.

## 💼 Predicting the List-to-Sale Discount Directly

For QCP, the most actionable number is often not the sale price but the
expected gap between list and sale. If we could predict that gap for each
listing, it would inform the offer. The honest benchmarks are again simple
rules: predict no change (sale = list), or predict the market's historical
median change. A model has to beat those to be useful.

Compute the discount target and fit a discount-prediction model. We leave the
list price level out of the features, so the model has to rely on the home's
characteristics, the market priors, and the description.

In [28]:
y_disc_train = (y_train_raw - X_train["list_price"]) / X_train["list_price"]
y_disc_test = (y_test_raw - X_test["list_price"]) / X_test["list_price"]

disc_pipe = make_pipeline(
    [c for c in NUMERIC_FEATURES if c != "list_price"],  # exclude raw list price level
    CATEGORICAL_FEATURES,
)
disc_pipe.fit(X_train, y_disc_train)
disc_pred = disc_pipe.predict(X_test)

In [29]:
def disc_metrics(y_true, y_pred):
    return {
        "MAE (pp)": mean_absolute_error(y_true, y_pred) * 100,
        "RMSE (pp)": np.sqrt(mean_squared_error(y_true, y_pred)) * 100,
        "R2": r2_score(y_true, y_pred),
    }


disc_comparison = pd.DataFrame(
    {
        "Naive rule: no change": disc_metrics(y_disc_test, np.zeros(len(y_disc_test))),
        "Market rule: market median change": disc_metrics(
            y_disc_test, X_test["market_hist_disc"] / 100
        ),
        "Discount model (XGBoost)": disc_metrics(y_disc_test, disc_pred),
    }
).T
disc_comparison.round(3)

,MAE (pp),RMSE (pp),R2
Naive rule: no change,5.245,7.859,-0.085
Market rule: market median change,5.034,7.427,0.031
Discount model (XGBoost),4.982,7.110,0.112


In [30]:
fig = px.scatter(
    pd.DataFrame({"actual": y_disc_test * 100, "predicted": disc_pred * 100}),
    x="actual",
    y="predicted",
    opacity=0.4,
    title="Predicted vs. actual list-to-sale discount (%)",
    labels={
        "actual": "Actual list-to-sale discount (%)",
        "predicted": "Predicted list-to-sale discount (%)",
    },
)
fig.add_shape(type="line", x0=-30, y0=-30, x1=30, y1=30, line=dict(dash="dash"))
fig.update_xaxes(range=[-30, 30])
fig.update_yaxes(range=[-30, 30])
fig.show()


The discount model explains only about 11% of the variation in the
list-to-sale change (R² about 0.11). Its average error is about 5.0
percentage points: barely better than the market rule (also about 5.0) and a
little better than assuming no discount (about 5.2). Most of the gap between
list and sale price likely depends on things we cannot see at list time, such
as how the asking price compares with the home's true value and how the
negotiation goes. Use market-level discounts as a starting point for offers;
this model is not precise enough to set a per-listing offer.

## 🧠 Investment-Committee Takeaways (Part 3)

What the price history shows, based on the outputs above:

- Across the footprint, a plausible sold cycle closes a median of about 55
  days after listing, from about 40 days in Davis to about 65 days in
  Gainesville and Savoy.
- The typical home sells about 1.6% below its list price. Gainesville (about
  -3%) and Athens (about -2.5%) show the deepest typical discounts; Madison
  sells at list at the median.
- Listings with a price cut close less often (about 77% vs. 87% for cycles
  with no cut), but additional cuts beyond the first do not lower the rate
  further.
- Knowing the list price cuts the sale-price error from about 12% (MAPE) to
  under 6%, even with the naive rule "sale = list". The best models reach
  about 5.3%, and for the typical listing the market-adjusted rule does
  better than either list-price model.
- The list-to-sale discount itself is hard to predict at list time (R² about
  0.11). Market-level discounts are a sensible starting point for offers; a
  per-listing discount model is not.

This concludes the four-credit case study. Your final deliverable should now
include the full pipeline (EDA -> baseline AVM -> NLP-augmented AVM ->
price-history analysis) and the slide deck synthesizing it for the
Investment Committee.